In [1]:
from databricks.connect import DatabricksSession
from pyspark.sql import functions as F

In [2]:
spark = DatabricksSession.builder.getOrCreate()

In [3]:
silver_df = spark.table('workspace.silver.job_postings')

In [8]:
silver_df.show()

+--------------------+--------------------+--------------------+--------------------+---------------+---------+----------+--------------------+-------+----------+----------+-------------+--------------------+-------------------+------------+--------------------+--------------------+--------------------+------------------------------+-------------------------+
|              job_id|           job_title|       employer_name|       employer_logo|employment_type|is_remote|      city|               state|country|min_salary|max_salary|salary_period|     job_description|          posted_at|   publisher|          apply_link|        _ingested_at|              skills|extracted_min_years_experience|extracted_education_level|
+--------------------+--------------------+--------------------+--------------------+---------------+---------+----------+--------------------+-------+----------+----------+-------------+--------------------+-------------------+------------+--------------------+--------------

In [ ]:
# gold layer skills demand
skills_df = silver_df.withColumn('skill', F.explode(F.col('skills')))

gold_skill_demand = skills_df.groupBy('skill').agg(F.count('*').alias('posting_count')).orderBy(F.desc('posting_count'))

In [7]:
gold_skill_demand.show()

+----------------+-------------+
|           skill|posting_count|
+----------------+-------------+
|             SQL|           18|
|         ETL/ELT|           16|
|    Data Quality|           13|
|    Apache Spark|           12|
|          Python|           11|
|   Data Modeling|           11|
| Data Governance|           11|
|      Databricks|           10|
|Data Warehousing|            9|
|             AWS|            9|
|     Agile/Scrum|            8|
|        Redshift|            8|
|           CI/CD|            8|
|             Git|            8|
|             GCP|            7|
|       Snowflake|            6|
|        BigQuery|            6|
|              S3|            5|
|           Azure|            5|
|         Airflow|            4|
+----------------+-------------+
only showing top 20 rows


In [20]:
# gold layer experience requirements
experience_range = silver_df.withColumn('experience_range',
    F.when(F.col('extracted_min_years_experience') <= 2.0, '0-2')
     .when(F.col('extracted_min_years_experience') <= 5.0, '3-5')
     .when(F.col('extracted_min_years_experience') <= 8.0, '5-8')
     .when(F.col('extracted_min_years_experience') >= 9.0, '10+')
     .otherwise('Not Found')
)

gold_experience = experience_range.groupBy('experience_range').count()

In [21]:
gold_experience.show()

+----------------+-----+
|experience_range|count|
+----------------+-----+
|             0-2|    2|
|       Not Found|    6|
|             3-5|    7|
|             5-8|    5|
+----------------+-----+



In [ ]:
# postings by location and salary